# Purpose

Calculate h_a for different combinations of Avobenzone and NPS absorber concentrations and various source spectra, including:

- 365 nm 
    - with 370 nm short pass filter
- 385 nm
    - full spectrum
    - 400 nm short pass filter
    - 390-400 nm band pass filter
- 405 nm
    - full spectrum
    - short pass filter
    - band pass filter
    - Also try Sudan I with Avobenzone?
    
Use the Asahi calculated transmission data from 2021 for the bandpass filter.

Develop source spectrum manipulation tools to do the following:

- Shift spectrum left or right while filling in data on shifted end of spectrum so full wavelength range is available.
- Define filter transmission properties as a spectrum.
- Apply a filter to another spectrum (multiply two spectra--the filter transmission and the source spectrum).
    - Short pass
    - Band pass
- Define some filters
    - 370 nm short pass
    - Asahi calculated band pass
        - 10 nm band pass (380 nm with 10 nm width at 1% transmission bandwidth)
        - 15 nm band pass (360 nm with 15 nm width at 1% transmission bandwidth)
    - Constructed band pass

In [2]:
import copy
from pathlib import Path
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# %matplotlib widget

from scipy.interpolate import interp1d
import scipy
print(scipy.__version__)

import spectra.data as data
from spectra.resin import ResinConstituentAbsorber, ResinConstituentMonomer, Resin
from spectra.irradiance import Irradiance
from spectra.utilities import ArrayParams, find_index_of_nearest, find_index_of_max, shift, shift_peak_wavelength
from spectra.normalizeddose import NormalizedDose
from spectra.sourcespectrum import SourceSpectrum

1.8.1


In [3]:
help(data)

Help on package spectra.data in spectra:

NAME
    spectra.data

PACKAGE CONTENTS
    absorbers (package)
    sources (package)

DATA
    absorbers = {'avobenzone_in_PEGDA_2017-04-13': <spectra.absorberspectr...
    absorbers_directory = PosixPath('/Users/nordin/Documents/Projects/3D_p...
    material_abbreviations = {'Avobenzone': 'Avo', 'BLS99-2': 'B99', 'BME'...
    monomers = {'HDDA': Monomer(/Users/nordin/Documents/Projects/3D_prin.....
    monomers_directory = PosixPath('/Users/nordin/Documents/Projects/3D_p....
    photoinitiators = {'irgacure819_in_PEGDA_2017-04-13': <spectra.absorbe...
    photoinitiators_directory = PosixPath('/Users/nordin/Documents/Project...
    sources = {'365nm_visitech_with_50mm_asahi_filter_2018-11-22': <spectr...
    sources_directory = PosixPath('/Users/nordin/Documents/Projects/3D_p.....

FILE
    /Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/__init__.py




In [5]:
data.sources

{'365nm_visitech_with_50mm_asahi_filter_2018-11-22': <spectra.sourcespectrum.SourceSpectrum at 0x7ff3623965e0>,
 'Asiga_405nm_100um_fiber-2016-12-14': <spectra.sourcespectrum.SourceSpectrum at 0x7ff3623967f0>,
 'HR1_385nm_100um_fiber-2016-12-14': <spectra.sourcespectrum.SourceSpectrum at 0x7ff362396940>,
 'HR2_385nm_2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7ff3623969a0>,
 'HR3.2_365nm_370nm_short_pass_filter-2019-12-31': <spectra.sourcespectrum.SourceSpectrum at 0x7ff362396a00>,
 'HR3.2_365nm_no_filter-2019-12-31': <spectra.sourcespectrum.SourceSpectrum at 0x7ff362396a60>,
 'HR3.3_365nm_370nm_short_pass_filter-2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7ff362396ac0>,
 'HR3.3_365nm_no_filter-2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7ff362396b20>,
 'HR3.3_365nm_through_tray_370nm_short_pass_filter-2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7ff362396b80>,
 'HR3.3_365nm_through_tray_no_filter-2020-03-23': <spectra.sourcespectrum.S